# 10 — Freddie baseline model development and freeze

**Scope:** Fit candidates on eligible Freddie 2015–2016 loans, select on 2017, then refit each selected model on 2015–2017. Do not inspect 2018–2023 outcomes or fit preprocessing on them here. The frozen models will be evaluated by a subsequent notebook using half-year cohorts.

Protocol: Step 5 Model Development and Frozen Evaluation Protocol (30 September 2026). Target: existing `TARGET_COMPOSITE_CREDIT_EVENT`, 24-month horizon. Original 29 predictors from `src/freddie_features.py`. No SMOTE, class weighting, scale_pos_weight, probability recalibration, or binary threshold selection.

** This notebook needs `scikit-learn`, `xgboost`, and `joblib`; see updated requirements. Model artifacts are local generated outputs and should not be committed to Git.


In [2]:
from pathlib import Path
import sys, json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn, xgboost, joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

ROOT=Path.cwd().parent if Path.cwd().name.lower()=='notebooks' else Path.cwd()
assert (ROOT/'src'/'freddie_features.py').is_file(), 'Open the repository root or its notebooks directory.'
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.freddie_features import (PREDICTOR_COLUMNS, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    REQUIRED_SOURCE_COLUMNS, build_engineered_features, fit_preprocessing_contract,
    apply_preprocessing_contract)

DATA=ROOT/'data'/'processed'/'Freddie_Mac'
INPUT=DATA/'freddie_modeling_dataset_2015_2023_24m.parquet'
SAVED_CONTRACT=DATA/'freddie_feature_contract.json'
OUT=DATA/'model_baselines_2015_2017'
assert INPUT.is_file() and SAVED_CONTRACT.is_file(), 'Run notebook 07 and verify the modeling dataset paths first.'
assert len(PREDICTOR_COLUMNS)==29 and len(NUMERIC_FEATURES)==15 and len(CATEGORICAL_FEATURES)==14
TARGET='TARGET_COMPOSITE_CREDIT_EVENT'
SEED=42
print('Input:',INPUT)


Input: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m.parquet


## 1. Rebuild raw engineered features for the internal validation split

Notebook 07's saved feature parquet already uses a 2015–2017 preprocessing contract; using it to select a model against 2017 would leak validation statistics. Read the validated modeling dataset and rebuild origination-time features, then fit the contract on 2015–2016 only. Hold 2017 untouched until scoring.


In [3]:
raw=pd.read_parquet(INPUT,columns=list(dict.fromkeys([*REQUIRED_SOURCE_COLUMNS,'HORIZON_MONTHS'])))
assert set(raw['SOURCE YEAR'].dropna().astype(int))==set(range(2015,2024))
assert not raw.duplicated(['SOURCE YEAR','LOAN IDENTIFIER']).any()
assert set(pd.to_numeric(raw['HORIZON_MONTHS']).unique())=={24}
development=raw.loc[raw['SOURCE YEAR'].isin([2015,2016,2017])].copy()
del raw
engineered=build_engineered_features(development)
assert list(engineered.columns).count(TARGET)==1
assert not engineered[TARGET].isna().any()
assert set(engineered[TARGET].astype(int).unique())=={0,1}
fit_mask=engineered['SOURCE YEAR'].isin([2015,2016]).to_numpy()
val_mask=(engineered['SOURCE YEAR']==2017).to_numpy()
assert fit_mask.sum()>0 and val_mask.sum()>0
internal_contract=fit_preprocessing_contract(engineered,training_years=(2015,2016))
internal=apply_preprocessing_contract(engineered,internal_contract)
X_fit=internal.loc[fit_mask,PREDICTOR_COLUMNS].copy()
X_val=internal.loc[val_mask,PREDICTOR_COLUMNS].copy()
y_fit=internal.loc[fit_mask,TARGET].astype(int).to_numpy()
y_val=internal.loc[val_mask,TARGET].astype(int).to_numpy()
assert set(y_fit)=={0,1} and set(y_val)=={0,1}
assert not X_fit.isna().any().any() and not X_val.isna().any().any()
print('Fit:',len(X_fit),'events:',int(y_fit.sum()),'Validation:',len(X_val),'events:',int(y_val.sum()))


Fit: 95320 events: 572 Validation: 48278 events: 391


## 2. Predeclared candidates and 2017 selection

The one-hot encoder and numeric scaler are fitted inside each candidate pipeline on 2015–2016. Both model families use the same frozen 29 input definitions. The logistic model standardizes numeric columns; XGBoost passes them through. The grid and AP/Brier tie rule are fixed in the Step 5 protocol. No 2018+ labels enter selection.


In [4]:
def model_pipeline(family, params):
    numeric=Pipeline([('scale',StandardScaler())]) if family=='logistic' else 'passthrough'
    preparation=ColumnTransformer([
        ('numeric',numeric,NUMERIC_FEATURES),
        ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=True),CATEGORICAL_FEATURES),
    ],remainder='drop',sparse_threshold=0.3)
    if family=='logistic':
        model=LogisticRegression(C=params['C'],penalty='l2',class_weight=None,
                                 max_iter=2000,random_state=SEED)
    else:
        model=XGBClassifier(objective='binary:logistic',max_depth=params['max_depth'],
                            n_estimators=params['n_estimators'],learning_rate=params['learning_rate'],
                            subsample=0.8,colsample_bytree=0.8,reg_lambda=1,
                            eval_metric='logloss',random_state=SEED,n_jobs=4,
                            tree_method='hist',scale_pos_weight=1)
    return Pipeline([('prepare',preparation),('model',model)])

GRID={'logistic':[{'C':c} for c in (0.1,1.0,10.0)],
      'xgboost':[{'max_depth':d,'n_estimators':n,'learning_rate':lr}
                 for d,n,lr in ((2,200,0.05),(3,200,0.05),(3,350,0.05))]}
results=[]
for family,candidates in GRID.items():
    for rank,params in enumerate(candidates):
        pipe=model_pipeline(family,params)
        pipe.fit(X_fit,y_fit)
        p=pipe.predict_proba(X_val)[:,1]
        assert np.isfinite(p).all() and ((0<=p)&(p<=1)).all()
        result={'family':family,'candidate_order':rank,'params':json.dumps(params,sort_keys=True),
                'n_fit':int(len(y_fit)),'positives_fit':int(y_fit.sum()),
                'n_validation':int(len(y_val)),'positives_validation':int(y_val.sum()),
                'prevalence_validation':float(y_val.mean()),'mean_predicted_validation':float(p.mean()),
                'average_precision':float(average_precision_score(y_val,p)),
                'roc_auc':float(roc_auc_score(y_val,p)),
                'brier':float(brier_score_loss(y_val,p))}
        results.append(result)
        print(family,params,'AP',round(result['average_precision'],5),'Brier',round(result['brier'],6))
table=pd.DataFrame(results)

def select_candidate(frame):
    best_ap=frame['average_precision'].max()
    close=frame.loc[(best_ap-frame['average_precision'])<0.005].copy()
    # Brier priority among candidates within 0.005 AP; six-decimal Brier ties use simple-grid order.
    close['brier_6']=close['brier'].round(6)
    return close.sort_values(['brier_6','candidate_order'],kind='stable').iloc[0]

chosen={family:select_candidate(table.loc[table.family==family]).to_dict()
        for family in GRID}
display(table.drop(columns=['params']).round(6))
print('Selected:',{k:json.loads(v['params']) for k,v in chosen.items()})


e:\capstone_project\mortgage-model-reliability\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic {'C': 0.1} AP 0.04992 Brier 0.007867


e:\capstone_project\mortgage-model-reliability\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic {'C': 1.0} AP 0.04802 Brier 0.007877


e:\capstone_project\mortgage-model-reliability\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic {'C': 10.0} AP 0.04697 Brier 0.007881
xgboost {'max_depth': 2, 'n_estimators': 200, 'learning_rate': 0.05} AP 0.04768 Brier 0.007868
xgboost {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.05} AP 0.04873 Brier 0.007866
xgboost {'max_depth': 3, 'n_estimators': 350, 'learning_rate': 0.05} AP 0.04429 Brier 0.007883


,family,candidate_order,n_fit,positives_fit,n_validation,positives_validation,prevalence_validation,mean_predicted_validation,average_precision,roc_auc,brier
0,logistic,0,95320,572,48278,391,0.008099,0.007615,0.049916,0.842167,0.007867
1,logistic,1,95320,572,48278,391,0.008099,0.007498,0.048019,0.839049,0.007877
2,logistic,2,95320,572,48278,391,0.008099,0.007409,0.046966,0.836678,0.007881
3,xgboost,0,95320,572,48278,391,0.008099,0.007231,0.047680,0.835425,0.007868
4,xgboost,1,95320,572,48278,391,0.008099,0.007176,0.048730,0.832994,0.007866
5,xgboost,2,95320,572,48278,391,0.008099,0.007159,0.044287,0.827300,0.007883


Selected: {'logistic': {'C': 0.1}, 'xgboost': {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}}


## 3. Final 2015–2017 refit and freeze

Verify the saved notebook 07 contract is exactly the one regenerated from the source on all 2015–2017 loans. Then refit the selected pipelines using only those loans. Store a model manifest with input schema, model settings, development counts, source fingerprint, and library versions. The frozen artifacts will be read by the later out-of-time evaluation notebook.


In [5]:
with SAVED_CONTRACT.open(encoding='utf-8') as f: saved_contract=json.load(f)
final_contract=fit_preprocessing_contract(engineered,training_years=(2015,2016,2017))
assert saved_contract==final_contract, 'Saved 2015–2017 contract differs; reconcile notebook 07 before fitting.'
final=apply_preprocessing_contract(engineered,saved_contract)
X_all=final.loc[:,PREDICTOR_COLUMNS].copy()
y_all=final[TARGET].astype(int).to_numpy()
assert not X_all.isna().any().any() and set(y_all)=={0,1}
OUT.mkdir(parents=True,exist_ok=True)
table.to_csv(OUT/'development_2017_candidate_metrics.csv',index=False)
models={}
for family,row in chosen.items():
    params=json.loads(row['params'])
    pipeline=model_pipeline(family,params)
    pipeline.fit(X_all,y_all)
    artifact=OUT/f'freddie_{family}_2015_2017.joblib'
    joblib.dump(pipeline,artifact)
    loaded=joblib.load(artifact)
    p1=pipeline.predict_proba(X_all.head(50))[:,1]
    p2=loaded.predict_proba(X_all.head(50))[:,1]
    np.testing.assert_allclose(p1,p2,rtol=1e-12,atol=1e-12)
    models[family]={'selected_params':params,'validation_average_precision':row['average_precision'],
                    'validation_brier':row['brier'],'artifact':artifact.name,
                    'sha256':hashlib.sha256(artifact.read_bytes()).hexdigest()}
contract_path=OUT/'freddie_feature_contract.json'
contract_path.write_text(json.dumps(saved_contract,indent=2),encoding='utf-8')
with INPUT.open('rb') as input_file:
    input_sha256=hashlib.file_digest(input_file,'sha256').hexdigest()
manifest={'protocol':'Step 5, 2026-09-30','target':TARGET,'horizon_months':24,
          'development_years':[2015,2016,2017],'internal_fit_years':[2015,2016],
          'internal_validation_year':2017,'predictors':PREDICTOR_COLUMNS,
          'numeric_predictors':NUMERIC_FEATURES,'categorical_predictors':CATEGORICAL_FEATURES,
          'development_rows':int(len(y_all)),'development_events':int(y_all.sum()),
          'development_prevalence':float(y_all.mean()),
          'input_file_sha256':input_sha256,
          'feature_contract_sha256':hashlib.sha256(contract_path.read_bytes()).hexdigest(),
          'models':models,'python':platform.python_version(),'pandas':pd.__version__,
          'numpy':np.__version__,'scikit_learn':sklearn.__version__,
          'xgboost':xgboost.__version__,'joblib':joblib.__version__,
          'out_of_time_scored':False}
(OUT/'model_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Frozen development artifacts:',OUT)
print('Rows and events:',manifest['development_rows'],manifest['development_events'])


e:\capstone_project\mortgage-model-reliability\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Frozen development artifacts: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\model_baselines_2015_2017
Rows and events: 143598 963


## Review gate

Upload only `development_2017_candidate_metrics.csv` and `model_manifest.json` for review. Keep joblib model files and raw/modeling data local. This notebook does **not** report 2018–2023 performance, population drift, or Fannie Mae validation. Its 2017 selection is retrospective: 2015–2017 complete 24-month labels were not all available at the start of 2018. The separate historical retraining exercise requires an as-of-date label-availability test.
